# Évaluation Automatique du RAG avec DeepEval
Ce notebook automatise l'évaluation scientifique de votre pipeline RAG en utilisant le paradigme **LLM-as-a-Judge** via un modèle local (Ollama) et le framework **DeepEval**. 

**Méthodologie** : Il compare les réponses générées par le système aux réponses du *Gold Standard* (vérité terrain) selon deux métriques principales :
- **Faithfulness (Fidélité)** : Vérifie l'absence d'hallucinations par rapport au contexte fourni.
- **Answer Relevance (Pertinence)** : Vérifie si la réponse est directe et utile par rapport à la question posée.

In [ ]:
!pip install deepeval pandas openpyxl requests

## 1. Configuration du Juge (Ollama + DeepEval)
Nous créons une classe personnalisée pour permettre à DeepEval d'utiliser notre LLM local hébergé sur Ollama (ex: `mistral:7b` ou `qwen2.5:7b`) comme évaluateur, afin d'éviter les coûts d'API externes.

In [ ]:
import os
import time
import pandas as pd
import requests
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.metrics import FaithfulnessMetric, AnswerRelevanceMetric
from deepeval.test_case import LLMTestCase

class OllamaEvaluationJudge(DeepEvalBaseLLM):
    def __init__(self, model_name="mistral:7b"):
        self.model_name = model_name

    def load_model(self):
        return self.model_name

    def generate(self, prompt: str) -> str:
        try:
            response = requests.post(
                "http://localhost:11434/api/generate",
                json={"model": self.model_name, "prompt": prompt, "stream": False},
                timeout=120
            )
            return response.json().get("response", "")
        except Exception as e:
            return f"Error connecting to Ollama: {str(e)}"

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return self.model_name

# Initialisation du modèle Juge (Assurez-vous qu'Ollama est lancé en arrière-plan)
eval_judge = OllamaEvaluationJudge(model_name="mistral:7b")
print("✅ Modèle Juge initialisé.")

## 2. Connexion à votre Système RAG
Ici, vous devez connecter la fonction de génération de votre projet. 
*Note : Si le module n'est pas trouvé, le script utilisera une fonction mock (simulation) pour vous montrer le fonctionnement.*

In [ ]:
try:
    import sys
    sys.path.append(os.path.abspath("../src")) # Ajustez le chemin vers votre dossier src/
    
    # À DÉCOMMENTER ET ADAPTER SELON VOTRE CODE :
    # from llm_chain import generate_rag_response
    # def votre_fonction_rag(question):
    #     reponse, chunks = generate_rag_response(question)
    #     return reponse, chunks
    
    raise ImportError # Forcer la simulation par défaut pour cet exemple
except ImportError:
    print("[⚠️] Importation du RAG non configurée. Utilisation de la fonction de simulation RAG.")
    def votre_fonction_rag(question):
        time.sleep(1.5) # Simule le temps de génération LLM
        reponse_simulee = "D'après le document, il faut obtenir au moins 5,5/10 pour valider le module."
        chunks_simules = ["[Article 3.2.2] La note minimale de validation d'un module est fixée à 5,5/10."]
        return reponse_simulee, chunks_simules

## 3. Lancement de la Boucle d'Évaluation sur le Gold Standard
Le script va lire chaque ligne de votre fichier de test, interroger le RAG, puis demander au modèle Juge d'évaluer la qualité de la réponse.

In [ ]:
input_excel = "Dataset_Evaluation_Gold_Standard_30Q.xlsx"

if not os.path.exists(input_excel):
    print(f"[❌] ERREUR : Le fichier {input_excel} est introuvable. Placez-le dans le même dossier que ce notebook.")
else:
    df_gold = pd.read_excel(input_excel)
    results_list = []
    print(f"[🚀] Démarrage de l'évaluation scientifique sur {len(df_gold)} questions...\n")

    for index, row in df_gold.iterrows():
        q_id = row['ID']
        niveau = row['Niveau_Complexite']
        question = row['Question']
        expected_output = row['Expected_Output']
        
        print(f"[{q_id}] Évaluation en cours ({niveau})...")
        
        # 1. Requête au RAG
        start_time = time.time()
        try:
            actual_output, retrieval_context_list = votre_fonction_rag(question)
        except Exception as e:
            actual_output = f"Erreur exécution RAG: {str(e)}"
            retrieval_context_list = ["Aucun contexte récupéré suite à une erreur."]
        elapsed_time = round(time.time() - start_time, 2)
        context_str = "\n---\n".join(retrieval_context_list)
        
        # 2. Préparation du cas de test DeepEval
        test_case = LLMTestCase(
            input=question,
            actual_output=actual_output,
            expected_output=expected_output,
            retrieval_context=retrieval_context_list
        )
        
        # 3. Évaluation par le Juge (Seuil à 0.75)
        faithfulness_metric = FaithfulnessMetric(threshold=0.75, model=eval_judge, include_reason=True)
        relevance_metric = AnswerRelevanceMetric(threshold=0.75, model=eval_judge, include_reason=True)
        
        try:
            faithfulness_metric.measure(test_case)
            f_score = round(faithfulness_metric.score, 2)
            f_reason = faithfulness_metric.reason
        except:
            f_score, f_reason = 0.0, "Échec d'évaluation de la métrique"
            
        try:
            relevance_metric.measure(test_case)
            r_score = round(relevance_metric.score, 2)
            r_reason = relevance_metric.reason
        except:
            r_score, r_reason = 0.0, "Échec d'évaluation de la métrique"
        
        passed = "OUI" if (f_score >= 0.75 and r_score >= 0.75) else "NON"
        compiled_reason = f"Fidélité ({f_score}): {f_reason} | Pertinence ({r_score}): {r_reason}"
        
        results_list.append({
            "ID": q_id,
            "Niveau": niveau,
            "Question": question,
            "Expected_Output": expected_output,
            "Actual_Output": actual_output,
            "Retrieved_Context": context_str,
            "Response_Time_Sec": elapsed_time,
            "Faithfulness_Score": f_score,
            "Answer_Relevance_Score": r_score,
            "DeepEval_Reason": compiled_reason,
            "Test_Passed": passed
        })
        
    print("\n[✅] Phase d'évaluation LLM terminée.")

## 4. Exportation et Stylisation du Fichier de Résultats
Ce bloc sauvegarde vos résultats dans un nouveau fichier Excel (`Resultats_Evaluation_RAG_DeepEval.xlsx`) en lui appliquant une mise en forme professionnelle, prête à être analysée pour votre mémoire.

In [ ]:
if 'results_list' in locals() and len(results_list) > 0:
    df_res = pd.DataFrame(results_list)
    output_excel = "Resultats_Evaluation_RAG_DeepEval.xlsx"
    df_res.to_excel(output_excel, index=False)

    # Application de la charte graphique avec openpyxl
    wb = openpyxl.load_workbook(output_excel)
    ws = wb.active
    ws.title = "Résultats DeepEval"

    header_fill = PatternFill(start_color="1F384B", end_color="1F384B", fill_type="solid")
    zebra_fill = PatternFill(start_color="F5F7F8", end_color="F5F7F8", fill_type="solid")
    white_fill = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
    green_alert = PatternFill(start_color="D1E7DD", end_color="D1E7DD", fill_type="solid")
    red_alert = PatternFill(start_color="F8D7DA", end_color="F8D7DA", fill_type="solid")

    font_header = Font(name="Arial", size=10, bold=True, color="FFFFFF")
    font_body = Font(name="Arial", size=9, color="222222")
    font_metrics = Font(name="Consolas", size=9, bold=True)
    
    thin_border = Border(
        left=Side(style='thin', color='D6DBDF'), right=Side(style='thin', color='D6DBDF'), 
        top=Side(style='thin', color='D6DBDF'), bottom=Side(style='thin', color='D6DBDF')
    )
    
    align_center = Alignment(horizontal="center", vertical="center", wrap_text=True)
    align_left = Alignment(horizontal="left", vertical="top", wrap_text=True)

    for col_idx in range(1, ws.max_column + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.fill = header_fill
        cell.font = font_header
        cell.alignment = align_center
        cell.border = thin_border
    ws.row_dimensions[1].height = 30

    for row_idx in range(2, ws.max_row + 1):
        current_fill = zebra_fill if row_idx % 2 == 0 else white_fill
        for col_idx in range(1, ws.max_column + 1):
            cell = ws.cell(row=row_idx, column=col_idx)
            cell.font = font_body
            cell.fill = current_fill
            cell.border = thin_border
            
            if col_idx in [1, 2]: 
                cell.alignment = align_center
            elif col_idx in [3, 4, 5, 6, 10]:
                cell.alignment = align_left
            elif col_idx in [7, 8, 9]:
                cell.font = font_metrics
                cell.alignment = align_center
            elif col_idx == 11:
                cell.alignment = align_center
                if cell.value == "OUI":
                    cell.fill = green_alert
                    cell.font = Font(name="Arial", size=9, bold=True, color="0F5132")
                else:
                    cell.fill = red_alert
                    cell.font = Font(name="Arial", size=9, bold=True, color="842029")
        ws.row_dimensions[row_idx].height = 65

    col_widths = {"A": 6, "B": 15, "C": 30, "D": 35, "E": 35, "F": 40, "G": 10, "H": 12, "I": 12, "J": 40, "K": 10}
    for col_letter, width in col_widths.items():
        ws.column_dimensions[col_letter].width = width
        
    ws.freeze_panes = "A2"
    wb.save(output_excel)
    print(f"\n[🎉] Fichier final généré avec succès : {output_excel}")
